# Knowledge Graph Pipeline1: Proof-of-Concept (without ODKE+)

This notebook implements a pipeline for financial knowledge extraction using FNSPID dataset.

In [1]:
!pip install neo4j beautifulsoup4 requests google-generativeai python-dotenv google langchain_neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.0/329.0 kB 24.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.


In [10]:
from huggingface_hub import hf_hub_download
import pandas as pd
import os
from google.colab import userdata
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, List, Type, TypeVar, Generic
import enum
from datetime import datetime
import requests
from bs4 import BeautifulSoup
from google import genai
from google.genai import types
import json
from neo4j import GraphDatabase, Session, Transaction
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph
from typing import Dict, List, Any

In [3]:
def join_string(item):
    Date, Article_title, Stock_symbol, Url, Lexrank_summary = item
    final_string = ""

    # Check if column has a unique value
    if pd.notna(Date):
        final_string += f"Date: {Date}"

    if pd.notna(Article_title):
        if final_string:
            final_string += " | "
        final_string += f"Article Title: {Article_title}"

    if pd.notna(Stock_symbol):
        if final_string:
            final_string += " | "
        final_string += f"Stock Symbol: {Stock_symbol}"

    if pd.notna(Url):
        if final_string:
            final_string += " | "
        final_string += f"Url: {Stock_symbol}"

    if pd.notna(Lexrank_summary):
        if final_string:
            final_string += " | "
        final_string += f"Summary: {Lexrank_summary}"

    return final_string

def filtered_text(file_path):
    df = pd.read_csv(
        file_path,
        dtype=str,
        low_memory=False,
        encoding='utf-8',
        on_bad_lines='skip'  # Skip problematic lines
    )

    # DataFrame processing
    df['Date'] = pd.to_datetime(df['Date'])
    date_time = input("Date time to filter (e.g., 2023-12-01 00:00:00+00:00): ")
    ticker_symbol = input("Ticker symbol to filter (e.g., AAPL, AMZN, GOOGL): ")
    df_filtered = df[(df['Date'].dt.date == pd.to_datetime(date_time).date()) & (df['Stock_symbol'] == ticker_symbol)]

    # Create the 'information' column
    df_filtered['Information'] = df_filtered[
        ['Date', 'Article_title', 'Stock_symbol','Lexrank_summary']
    ].apply(join_string, axis=1)

    # Group all information and return as text
    df_grouped = df_filtered.groupby('Date')['Information'].apply(lambda x: '\n'.join(x)).reset_index()
    sample = df_grouped.head().loc[0, 0]['Information']

    return sample

In [4]:
def extract_entities_and_relationship(text: str, schema_class: Type[BaseModel]):
    prompt = f"""
    Extract all relevant information from the following text and populate the provided data schema.
    IMPORTANT: The schema requires a `ProvableFact` object for many facts.
    You must fill the `value` field (the fact) AND the `evidence` field (the text snippet from the source text
    that proves the fact).

    **Text to Analyze:**
    ---
    {text}
    ---

    # Few-Shot Example

    **Input:**
    Date: 2023-12-01 | Article Title: Best Blue Chip Stocks? | Stock Symbol: AAPL | Url: https://nasdaq.com/article/123
    Summary: Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics. Recently, shares of AAPL stock have gained by 9.29% due to holiday sales.

    **Output:**
    [
        {{
            "triplet_id": "1",
            "entity": {{
                "name": "Apple Inc.",
                "entity_type": "ORG"
            }},
            "relationship": {{
                "relationship": "Produces",
                "evidence": "Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics."
            }},
            "target": {{
                "name": "consumer electronics",
                "entity_type": "PRODUCT"
            }},
            "temporal_info": {{
                "date": "2023-12-01",
                "extraction_type": "default"
            }},
            "chunk_text": "Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics.",
            "chunk_id": null,
            "page_id": null,
            "date": "2023-12-01",
            "ticker": "AAPL",
            "source_url": "https://nasdaq.com/article/123"
        }},
        {{
            "triplet_id": "2",
            "entity": {{
                "name": "AAPL",
                "entity_type": "ORG"
            }},
            "relationship": {{
                "relationship": "Increases",
                "evidence": "shares of AAPL stock have gained by 9.29%"
            }},
            "target": {{
                "name": "Share Price",
                "entity_type": "FIN_METRIC"
            }},
            "temporal_info": {{
                "date": "2023-12-01",
                "extraction_type": "default"
            }},
            "chunk_text": "shares of AAPL stock have gained by 9.29%",
            "chunk_id": null,
            "page_id": null,
            "date": "2023-12-01",
            "ticker": "AAPL",
            "source_url": "https://nasdaq.com/article/123"
        }},
        {{
            "triplet_id": "3",
            "entity": {{
                "name": "AAPL",
                "entity_type": "ORG"
            }},
            "relationship": {{
                "relationship": "Stock_Rise_Due_To",
                "evidence": "shares of AAPL stock have gained by 9.29% due to holiday sales"
            }},
            "target": {{
                "name": "Holiday Sales",
                "entity_type": "EVENT"
            }},
            "temporal_info": {{
                "date": "2023-12-01",
                "extraction_type": "default"
            }},
            "chunk_text": "shares of AAPL stock have gained by 9.29% due to holiday sales.",
            "chunk_id": null,
            "page_id": null,
            "date": "2023-12-01",
            "ticker": "AAPL",
            "source_url": "https://nasdaq.com/article/123"
        }}
    ]

    """
    load_dotenv()
    client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json",
            response_schema=schema_class
        )
    )

    result = json.loads(response.text)
    return result

In [5]:
# ==============================================================================
# GROUNDER: Verify facts against source text
# ==============================================================================
from ontology import GrounderResponse, ProvenanceModel, ExtractionPackage

def is_triplet_grounded(triplet: dict) -> bool:
    """
    Checks with a lightweight LLM if a triplet relationship is supported by its evidence.
    Adapted from Original_ODKE+.ipynb for financial triplets.
    """
    # Handle both nested and flat structures
    if isinstance(triplet.get('entity'), dict):
        entity = triplet['entity']['name']
        relationship = triplet['relationship']['relationship']
        target = triplet['target']['name']
        evidence = triplet.get('chunk_text', triplet['relationship'].get('evidence', ''))
    else:
        entity = triplet.get('entity', '')
        relationship = triplet.get('relationship', '')
        target = triplet.get('target', '')
        evidence = triplet.get('source_text', '')

    if not evidence:
        return False

    # Create a fact statement from the triplet
    fact = f"{entity} {relationship} {target}"

    prompt = f"""
    Verify if the following fact can be inferred from the provided text snippet.
    The fact must be explicitly mentioned or directly logically derivable.

    Fact to verify:
    "{fact}"

    Can this fact be derived from the following snippet?:
    "{evidence}"
    """

    try:
        load_dotenv()
        client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
        response = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1,
                response_mime_type="application/json",
                response_schema=GrounderResponse
            )
        )
        if response.parsed:
            return response.parsed.is_grounded
        else:
            triplet_id = triplet.get('triplet_id', 'unknown')
            print(f"GROUNDER WARNING: Could not parse response for triplet {triplet_id}. Defaulting to 'False'.")
            return False
    except Exception as e:
        triplet_id = triplet.get('triplet_id', 'unknown')
        print(f"GROUNDER ERROR for triplet {triplet_id}: {e}. Defaulting to 'False'.")
        return False

def ground_triplets(triplets: list, source_url: str = "") -> list:
    """
    Takes a list of triplets and validates each one against its evidence.
    Returns only the grounded triplets.
    Adapted from Original_ODKE+.ipynb ground_package function.
    """
    print(f"\n{'='*50}\n🔬 STARTING GROUNDING PROCESS\n{'='*50}")
    if source_url:
        print(f"Source URL: {source_url}")
    print(f"Total triplets to ground: {len(triplets)}")

    grounded_triplets = []
    for triplet in triplets:
        triplet_id = triplet.get('triplet_id', 'unknown')

        # Extract entity, relationship, target based on structure
        if isinstance(triplet.get('entity'), dict):
            entity = triplet['entity']['name']
            relationship = triplet['relationship']['relationship']
            target = triplet['target']['name']
            evidence = triplet.get('chunk_text', triplet['relationship'].get('evidence', ''))
        else:
            entity = triplet.get('entity', '')
            relationship = triplet.get('relationship', '')
            target = triplet.get('target', '')
            evidence = triplet.get('source_text', '')

        if not evidence:
            print(f"--- ⚠️ GROUNDING SKIPPED: No evidence for triplet {triplet_id}. Removing.")
            continue

        is_grounded = is_triplet_grounded(triplet)
        if not is_grounded:
            print(f"--- ❌ GROUNDING FAILED: Triplet {triplet_id} '{entity} {relationship} {target}' will be removed.")
        else:
            print(f"--- ✅ GROUNDING PASSED: Triplet {triplet_id} '{entity} {relationship} {target}'")
            grounded_triplets.append(triplet)

    print(f"\n🔬 GROUNDING COMPLETED: {len(grounded_triplets)}/{len(triplets)} triplets passed grounding.")
    return grounded_triplets

In [6]:
# ==============================================================================
# INGESTOR & CORROBORATOR: Handle staleness and conflicts
# ==============================================================================
from neo4j import GraphDatabase

def _tx_corroborate_and_ingest_triplet(
    tx: Transaction,
    triplet: dict,
    source_url: str,
    retrieved_at: datetime,
    trust_score: float = 1.0
):
    """
    Executes the Corroborator logic for a single triplet in a single transaction.
    Adapted from Original_ODKE+.ipynb _tx_corroborate_and_ingest function.

    Logic:
    1. Check for existing active relationships from other sources
    2. Apply "Freshness > Trust" algorithm to determine winner
    3. Deactivate old relationships if new one wins
    4. Create/update relationship with is_active flag
    """
    # Handle both nested and flat structures
    if isinstance(triplet.get('entity'), dict):
        entity = triplet['entity']['name']
        entity_type = triplet['entity']['entity_type']
        relationship = triplet['relationship']['relationship']
        target = triplet['target']['name']
        target_type = triplet['target']['entity_type']
        source_text = triplet.get('chunk_text', triplet['relationship'].get('evidence', ''))
    else:
        entity = triplet.get('entity', '')
        entity_type = triplet.get('entity_type', '')
        relationship = triplet.get('relationship', '')
        target = triplet.get('target', '')
        target_type = triplet.get('target_type', '')
        source_text = triplet.get('source_text', '')

    date = triplet.get('date', '')
    ticker = triplet.get('ticker', '')

    # Sanitize relationship type for Neo4j (replace spaces and special chars with underscores)
    rel_type = relationship.replace(' ', '_').replace('-', '_')

    # Find existing active relationships of the same type from other sources
    query_find_old = f"""
    MATCH (a:Entity {{name: $entity}})-[r:{rel_type}]->(b:Entity {{name: $target}})
    WHERE r.is_active = true AND r.source_url <> $source_url
    RETURN r.retrieved_at AS old_ts, r.trust_score AS old_trust, r.source_url AS old_url
    LIMIT 1
    """

    try:
        result = tx.run(query_find_old, entity=entity, target=target, source_url=source_url)
        old_fact = result.single()
    except Exception as e:
        # If relationship type doesn't exist yet, no old fact
        old_fact = None

    # Determine if new fact is the winner (Freshness > Trust algorithm)
    is_candidate_winner = False
    if not old_fact:
        is_candidate_winner = True
    else:
        old_ts = old_fact.get('old_ts')
        old_trust = old_fact.get('old_trust', 0.0)

        if old_ts:
            if retrieved_at > old_ts:
                is_candidate_winner = True
            elif retrieved_at == old_ts:
                if trust_score >= old_trust:
                    is_candidate_winner = True
                else:
                    is_candidate_winner = False
            else:
                is_candidate_winner = False
        else:
            is_candidate_winner = True

    # Ensure entities exist
    tx.run("MERGE (a:Entity {name: $entity}) ON CREATE SET a.type = $entity_type",
           entity=entity, entity_type=entity_type)
    tx.run("MERGE (b:Entity {name: $target}) ON CREATE SET b.type = $target_type",
           target=target, target_type=target_type)

    # Ensure Source node exists
    tx.run("MERGE (s:Source {url: $url})", url=source_url)

    if is_candidate_winner:
        print(f"--- 🏆 CORROBORATOR: NEW wins for '{entity} {relationship} {target}'")

        # Deactivate all old relationships of this type between these entities
        deactivate_query = f"""
        MATCH (a:Entity {{name: $entity}})-[r:{rel_type}]->(b:Entity {{name: $target}})
        WHERE r.is_active = true
        SET r.is_active = false
        """
        try:
            tx.run(deactivate_query, entity=entity, target=target)
        except Exception:
            pass  # No existing relationships to deactivate

        # Create new active relationship
        query_create = f"""
        MATCH (a:Entity {{name: $entity}}), (b:Entity {{name: $target}}), (s:Source {{url: $url}})
        MERGE (a)-[r:{rel_type}]->(b)
        SET r.is_active = true,
            r.date = $date,
            r.ticker = $ticker,
            r.source_url = $url,
            r.source_text = $source_text,
            r.retrieved_at = $retrieved_at,
            r.trust_score = $trust_score
        MERGE (a)-[src_rel:FROM_SOURCE]->(s)
        SET src_rel.is_active = true,
            src_rel.retrieved_at = $retrieved_at,
            src_rel.trust_score = $trust_score
        """
        tx.run(query_create,
               entity=entity, target=target, url=source_url,
               date=date, ticker=ticker, source_text=source_text,
               retrieved_at=retrieved_at, trust_score=trust_score)
    else:
        print(f"--- 🛡️ CORROBORATOR: OLD wins for '{entity} {relationship} {target}'")
        # Still create the relationship but mark it as inactive
        query_create = f"""
        MATCH (a:Entity {{name: $entity}}), (b:Entity {{name: $target}}), (s:Source {{url: $url}})
        MERGE (a)-[r:{rel_type}]->(b)
        SET r.is_active = false,
            r.date = $date,
            r.ticker = $ticker,
            r.source_url = $url,
            r.source_text = $source_text,
            r.retrieved_at = $retrieved_at,
            r.trust_score = $trust_score
        MERGE (a)-[src_rel:FROM_SOURCE]->(s)
        SET src_rel.is_active = false,
            src_rel.retrieved_at = $retrieved_at,
            src_rel.trust_score = $trust_score
        """
        tx.run(query_create,
               entity=entity, target=target, url=source_url,
               date=date, ticker=ticker, source_text=source_text,
               retrieved_at=retrieved_at, trust_score=trust_score)

def _tx_ingest_triplets_package(tx: Transaction, triplets: list, source_url: str,
                                retrieved_at: datetime, trust_score: float = 1.0):
    """
    Executes the entire triplet ingestion in a single transaction.
    Adapted from Original_ODKE+.ipynb _tx_ingest_product_package function.
    """
    print(f"--- ⏳ STALENESS-CHECK: Deactivating old facts from {source_url}...")

    # Deactivate all old relationships from this source
    tx.run("""
    MATCH ()-[r]->()
    WHERE r.source_url = $url AND r.is_active = true
    SET r.is_active = false
    """, url=source_url)

    # Also deactivate FROM_SOURCE relationships
    tx.run("""
    MATCH ()-[r:FROM_SOURCE]->(s:Source {url: $url})
    WHERE r.is_active = true
    SET r.is_active = false
    """, url=source_url)

    # Process each triplet
    for triplet in triplets:
        try:
            _tx_corroborate_and_ingest_triplet(tx, triplet, source_url, retrieved_at, trust_score)
        except Exception as e:
            print(f"Error processing triplet {triplet.get('triplet_id', 'unknown')}: {e}")
            continue

def ingest_triplets_with_corroboration(kg, triplets: list, source_url: str = "",
                                       trust_score: float = 1.0):
    """
    Manager function: Ingests triplets with grounding and corroboration.
    Adapted from Original_ODKE+.ipynb ingest_product_package function.
    """
    if not triplets:
        print("No triplets to ingest.")
        return

    # Step 1: Ground the triplets
    grounded_triplets = ground_triplets(triplets, source_url)

    if not grounded_triplets:
        print("No grounded triplets to ingest after grounding process.")
        return

    # Step 2: Ingest with corroboration
    retrieved_at = datetime.now()

    print(f"\n{'='*50}\n📥 STARTING INGESTION WITH CORROBORATION\n{'='*50}")
    print(f"Source URL: {source_url}")
    print(f"Trust Score: {trust_score}")
    print(f"Triplets to ingest: {len(grounded_triplets)}")

    with kg._driver.session() as session:
        session.execute_write(_tx_ingest_triplets_package, grounded_triplets,
                             source_url, retrieved_at, trust_score)

    print(f"✅ Ingestion transaction completed for {len(grounded_triplets)} triplets.")

In [7]:
# ==============================================================================
# INFERENCE: Create inferred relationships based on active graph state
# ==============================================================================
def create_inferred_relationships(kg):
    """
    Creates inferred relationships ONLY between active nodes.
    Adapted from Original_ODKE+.ipynb create_inferred_relationships function.

    Example inference rules for financial knowledge graph:
    - If Company A Partners_With Company B (active), infer bidirectional relationship
    - If Company A Produces Product X and Product X Related_To Concept Y (both active),
      infer Company A Related_To Concept Y
    - If Person X Member_Of Organization Y and Organization Y Regulates Company Z (both active),
      infer Person X Related_To Company Z
    """
    print(f"\n{'='*50}\n🔮 PHASE 2: CREATE INFERRED RELATIONSHIPS\n{'='*50}")

    with kg._driver.session() as session:
        # Delete all old inferred relationships
        print("Deleting all old :INFERRED relationships...")
        session.run("MATCH ()-[r:INFERRED]->() DELETE r")

        # Inference Rule 1: Bidirectional Partners_With relationships
        # If A Partners_With B (active), infer B Partners_With A
        print("\n--- Inference Rule 1: Bidirectional Partners_With...")
        query1 = """
        MATCH (a:Entity)-[r:Partners_With]->(b:Entity)
        WHERE r.is_active = true
        AND NOT EXISTS((b)-[:Partners_With]->(a))
        MERGE (b)-[inf:INFERRED {inference_type: 'bidirectional_partners',
                                  source_entity: a.name,
                                  target_entity: b.name,
                                  inferred_at: datetime()}]->(a)
        RETURN count(inf) AS inferred_count
        """
        result1 = session.run(query1)
        summary1 = result1.single()
        if summary1:
            print(f"   → Created {summary1['inferred_count']} bidirectional Partners_With relationships")

        # Inference Rule 2: Transitive Produces relationships
        # If A Produces B and B Related_To C (both active), infer A Related_To C
        print("\n--- Inference Rule 2: Transitive Produces->Related_To...")
        query2 = """
        MATCH (a:Entity)-[r1:Produces {is_active: true}]->(b:Entity)
        MATCH (b)-[r2:Related_To {is_active: true}]->(c:Entity)
        WHERE NOT EXISTS((a)-[:Related_To]->(c))
        MERGE (a)-[inf:INFERRED {inference_type: 'transitive_produces_related',
                                 intermediate_entity: b.name,
                                 inferred_at: datetime()}]->(c)
        RETURN count(inf) AS inferred_count
        """
        result2 = session.run(query2)
        summary2 = result2.single()
        if summary2:
            print(f"   → Created {summary2['inferred_count']} transitive Produces->Related_To relationships")

        # Inference Rule 3: Member_Of -> Regulates transitive
        # If Person X Member_Of Org Y and Org Y Regulates Company Z (both active), infer X Related_To Z
        print("\n--- Inference Rule 3: Member_Of->Regulates transitive...")
        query3 = """
        MATCH (p:Entity {type: 'PERSON'})-[r1:Member_Of {is_active: true}]->(o:Entity {type: 'ORG_REG'})
        MATCH (o)-[r2:Regulates {is_active: true}]->(c:Entity {type: 'ORG'})
        WHERE NOT EXISTS((p)-[:Related_To]->(c))
        MERGE (p)-[inf:INFERRED {inference_type: 'member_regulates_transitive',
                                 intermediate_org: o.name,
                                 inferred_at: datetime()}]->(c)
        RETURN count(inf) AS inferred_count
        """
        result3 = session.run(query3)
        summary3 = result3.single()
        if summary3:
            print(f"   → Created {summary3['inferred_count']} Member_Of->Regulates transitive relationships")

        # Inference Rule 4: Stock impact inference
        # If Company A Increases Metric X and Metric X is FIN_METRIC related to Stock Y (both active),
        # infer Company A Affects_Stock Stock Y
        print("\n--- Inference Rule 4: Stock impact inference...")
        query4 = """
        MATCH (c:Entity {type: 'ORG'})-[r1:Increases {is_active: true}]->(m:Entity {type: 'FIN_METRIC'})
        MATCH (m)-[r2:Related_To {is_active: true}]->(s:Entity {type: 'FIN_INST'})
        WHERE (s.name CONTAINS c.name OR s.name CONTAINS c.ticker)
        AND NOT EXISTS((c)-[:Affects_Stock]->(s))
        MERGE (c)-[inf:INFERRED {inference_type: 'stock_impact',
                                 metric: m.name,
                                 inferred_at: datetime()}]->(s)
        RETURN count(inf) AS inferred_count
        """
        result4 = session.run(query4)
        summary4 = result4.single()
        if summary4:
            print(f"   → Created {summary4['inferred_count']} stock impact relationships")

        total_inferred = (summary1['inferred_count'] if summary1 else 0) + \
                        (summary2['inferred_count'] if summary2 else 0) + \
                        (summary3['inferred_count'] if summary3 else 0) + \
                        (summary4['inferred_count'] if summary4 else 0)

        print(f"\n✅ Total inferred relationships created: {total_inferred}")

    print(f"🔮 INFERENCE PROCESS COMPLETED.\n{'='*50}")

## Updated Pipeline with Grounder -> Ingestor & Inference

The pipeline now includes:
1. **Grounder**: Verifies each triplet against its evidence (chunk_text or relationship.evidence)
2. **Ingestor & Corroborator**: Handles staleness (deactivates old facts) and resolves conflicts using "Freshness > Trust" algorithm
3. **Inference**: Creates inferred relationships based on active graph state

**To use the new pipeline**, replace the old ingestion code in Cell 9 with:
```python
# Ground triplets first
source_url = triplets[0].get('source_url', '') if triplets else ''
grounded_triplets = ground_triplets(triplets, source_url)

# Ingest with corroboration (replaces simple add_relationship_to_neo4j)
trust_score = 1.0
ingest_triplets_with_corroboration(graph, grounded_triplets, source_url, trust_score)

# Create inferred relationships
create_inferred_relationships(graph)
```

In [8]:
def clear_kg(kg):
    with kg._driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")

def add_relationship_to_neo4j(kg, triplets):
    """
    Add relationships to Neo4j graph.
    Handles both nested FinancialTripletModel structure and flat structure for backward compatibility.
    """
    with kg._driver.session() as session:
        for triplet in triplets:
            # Handle nested FinancialTripletModel structure
            if isinstance(triplet.get('entity'), dict):
                # Nested structure (FinancialTripletModel format)
                entity_name = triplet['entity']['name']
                entity_type = triplet['entity']['entity_type']
                rel_type = triplet['relationship']['relationship']
                target_name = triplet['target']['name']
                target_type = triplet['target']['entity_type']
                source_text = triplet.get('chunk_text', triplet['relationship'].get('evidence', ''))
            else:
                # Flat structure (backward compatibility)
                entity_name = triplet['entity']
                entity_type = triplet['entity_type']
                rel_type = triplet['relationship']
                target_name = triplet['target']
                target_type = triplet['target_type']
                source_text = triplet.get('source_text', '')

            query = f"MERGE (a:Entity {{name: $source}}) " \
                    f"ON CREATE SET a.type = $source_type " \
                    f"MERGE (b:Entity {{name: $target}}) " \
                    f"ON CREATE SET b.type = $target_type " \
                    f"MERGE (a)-[r:{rel_type}]->(b) " \
                    f"SET r.date = $date, " \
                    f"r.ticker = $ticker, " \
                    f"r.source_url = $source_url, " \
                    f"r.source_text = $source_text"
            try:
                session.run(
                    query,
                    source=entity_name,
                    source_type=entity_type,
                    target=target_name,
                    target_type=target_type,
                    date=triplet.get('date', 'N/A'),
                    ticker=triplet.get('ticker', 'N/A'),
                    source_url=triplet.get('source_url', ''),
                    source_text=source_text
                )
            except Exception as e:
                print(f"Error adding relationship: {e}")

In [15]:
CYPHER_GENERATION_TEMPLATE = """
# Role
You are an expert in Neo4j Cypher query language and financial knowledge graphs. Your task is to convert natural language questions into precise Cypher queries.

# Knowledge Graph Schema

## Node Structure
- Label: Entity
- Properties:
  - name: string (entity name)
  - type: string (entity type from 24 categories)

## Entity Types (24 Categories)
**Core Business:** ORG, COMP, SEGMENT, PERSON
**Geographic & Regulatory:** GPE, ORG_GOV, ORG_REG
**Financial & Market:** FIN_INST, FIN_MARKET, FIN_METRIC, ECON_IND
**Products & Operations:** PRODUCT, CONCEPT, RAW_MATERIAL, LOGISTICS
**Risk & Compliance:** RISK_FACTOR, LITIGATION, REGULATORY_REQUIREMENT, ACCOUNTING_POLICY
**Strategic & ESG:** EVENT, SECTOR, ESG_TOPIC, MACRO_CONDITION, COMMENTARY

## Relationship Structure
- Dynamic relationship types (27 types): Has_Stake_In, Regulates, Operates_In, Announces, Introduces, Produces, Invests_In, Partners_With, Supplies, Impacts, Positively_Impacts, Negatively_Impacts, Increases, Decreases, Affects_Stock, Involved_In, Impacted_By, Faces, Depends_On, Discloses, Guides_On, Complies_With, Subject_To, Related_To, Member_Of, Causes_Shortage_Of, Stock_Decline_Due_To, Stock_Rise_Due_To, Market_Reacts_To
- Properties:
  - date: string
  - ticker: string
  - source_url: string
  - source_text: string

# CRITICAL: Always Include Source Information
Every query MUST return both source_url and source_text properties from relationships to enable proper verification.
Use this pattern: r.source_url AS SourceURL, r.source_text AS Evidence

# Few-Shot Examples

## Example 1: Direct Entity Query
**Question:** "What products does Apple produce?"
**Analysis:**
- Entity: Apple (ORG)
- Relationship: Produces
- Target Type: PRODUCT

**Cypher Query:**
```cypher
MATCH (org:Entity {{name: 'AAPL'}})-[r:Produces]->(product:Entity)
WHERE product.type = 'PRODUCT'
RETURN org.name AS Company,
       product.name AS Product,
       r.date AS Date,
       r.source_url AS SourceURL,
       r.source_text AS Evidence
```

**Explanation:** Find all entities connected to Apple through Produces relationship where target is a PRODUCT type. Returns source URL and text for verification.

**Expected Output:** List of products Apple produces with supporting evidence and source URLs.

## Example 2: Financial Metric Query
**Question:** "How much did AAPL stock increase?"
**Analysis:**
- Entity: AAPL (ORG)
- Relationship: Increases
- Target Type: FIN_METRIC

**Cypher Query:**
```cypher
MATCH (org:Entity)-[r:Increases]->(metric:Entity)
WHERE org.name CONTAINS 'AAPL' AND metric.type = 'FIN_METRIC'
RETURN org.name AS Entity,
       metric.name AS Metric,
       r.date AS Date,
       r.source_url AS SourceURL,
       r.source_text AS Evidence
ORDER BY r.date DESC
```

**Explanation:** Find financial metrics that increased, filtering by AAPL and metric type. Includes source URL for verification.

**Expected Output:** Percentage increases with dates, evidence text, and source URLs.

## Example 3: Relationship Discovery
**Question:** "Who are Apple's competitors?"
**Analysis:**
- Entity: Apple (ORG/COMP)
- Relationship: COMPETES_WITH or related competitive relationships
- Target Type: COMP/ORG

**Cypher Query:**
```cypher
MATCH (apple:Entity)-[r]-(competitor:Entity)
WHERE apple.name CONTAINS 'Apple'
  AND (competitor.type = 'COMP' OR competitor.type = 'ORG')
  AND type(r) IN ['COMPETES_WITH', 'Related_To']
RETURN apple.name AS Company,
       type(r) AS Relationship,
       competitor.name AS Competitor,
       r.date AS Date,
       r.source_url AS SourceURL,
       r.source_text AS Evidence
```

**Explanation:** Find entities related to Apple that are companies, checking multiple relationship types. Returns source URLs for verification.

**Expected Output:** List of competitor companies with relationship context and source URLs.

## Example 4: Event Impact Query
**Question:** "What events affected Apple's stock?"
**Analysis:**
- Entity: Apple (ORG)
- Relationship: Stock_Rise_Due_To, Stock_Decline_Due_To, Affects_Stock
- Target Type: EVENT

**Cypher Query:**
```cypher
MATCH (org:Entity)-[r]->(event:Entity)
WHERE org.name CONTAINS 'Apple'
  AND event.type = 'EVENT'
  AND type(r) IN ['Stock_Rise_Due_To', 'Stock_Decline_Due_To', 'Affects_Stock']
RETURN org.name AS Company,
       type(r) AS Impact,
       event.name AS Event,
       r.date AS Date,
       r.source_url AS SourceURL,
       r.source_text AS Evidence
ORDER BY r.date DESC
```

**Explanation:** Find all events that had stock impact on Apple, showing the type of impact. Includes source URLs for verification.

**Expected Output:** Events with their impact direction, supporting evidence, and source URLs.

## Example 5: Multi-Hop Query
**Question:** "What products does Apple produce and which events increased their sales?"
**Analysis:**
- Multi-step: Apple -> Produces -> Product, Product -> Increases -> EVENT
- Entity Types: ORG, PRODUCT, EVENT
- Relationships: Produces, Increases, Stock_Rise_Due_To

**Cypher Query:**
```cypher
MATCH (org:Entity {{name: 'Apple Inc.'}})-[r1:Produces]->(product:Entity)-[r2:Increases|Stock_Rise_Due_To]->(target:Entity)
WHERE product.type = 'PRODUCT' AND (target.type = 'EVENT' OR target.type = 'FIN_METRIC')
RETURN org.name AS Company,
       product.name AS Product,
       type(r2) AS Relationship,
       target.name AS Target,
       r2.date AS Date,
       r2.source_url AS SourceURL,
       r2.source_text AS Evidence
```

**Explanation:** Two-hop query connecting Apple to products to sales increases or events. Returns source information from the final relationship.

**Expected Output:** Products and the events/metrics that increased their performance with source URLs.

## Example 6: Aggregation Query
**Question:** "How many relationships does Apple have in the graph?"
**Analysis:**
- Count query
- Entity: Apple
- All relationships

**Cypher Query:**
```cypher
MATCH (org:Entity)-[r]-(other:Entity)
WHERE org.name CONTAINS 'Apple'
RETURN org.name AS Company,
       count(r) AS TotalRelationships,
       collect(DISTINCT type(r)) AS RelationshipTypes,
       collect(DISTINCT r.source_url)[0..5] AS SampleSourceURLs
```

**Explanation:** Count all relationships for Apple and list unique relationship types. Returns sample source URLs for reference.

**Expected Output:** Total count, list of relationship types, and sample source URLs.

## Example 7: Time-Based Query
**Question:** "What happened with Apple in December 2023?"
**Analysis:**
- Entity: Apple
- Time filter: December 2023
- All relationships

**Cypher Query:**
```cypher
MATCH (org:Entity)-[r]-(other:Entity)
WHERE org.name CONTAINS 'Apple'
  AND r.date CONTAINS '2023-12'
RETURN org.name AS Company,
       type(r) AS Relationship,
       other.name AS RelatedEntity,
       other.type AS EntityType,
       r.date AS Date,
       r.source_url AS SourceURL,
       r.source_text AS Evidence
ORDER BY r.date DESC
```

**Explanation:** Find all Apple-related activities in December 2023. Returns source URLs and evidence text for each relationship.

**Expected Output:** All relationships and entities for that time period with source URLs for verification.

# Instructions
1. Analyze the question to identify:
   - Key entities mentioned
   - Entity types involved
   - Relationship types needed
   - Any filters (date, ticker, metric type)
   - Whether aggregation is needed

2. Generate a Cypher query that:
   - Uses proper Neo4j syntax
   - Filters by entity types when relevant
   - Includes date/ticker filters if mentioned
   - **ALWAYS returns r.source_url AS SourceURL and r.source_text AS Evidence**
   - Returns meaningful column names
   - Orders results when appropriate

3. Provide:
   - The Cypher query
   - Brief explanation of the query logic
   - Description of expected output

# MANDATORY RETURN PATTERN
Every query MUST include these columns when relationships are involved:
- r.source_url AS SourceURL
- r.source_text AS Evidence

For multi-hop queries, include source information from the most relevant relationship (typically the final one).

# Output Format
Return ONLY a valid JSON object (no markdown formatting):
{{
    "cypher_query": "MATCH ...",
    "explanation": "Brief explanation of the query logic",
    "expected_output": "Description of what results will show"
}}

# Question to Convert
{question}
"""

def generate_cypher_query(question: str) -> Dict[str, Any]:
  load_dotenv()
  client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
  prompt = CYPHER_GENERATION_TEMPLATE.format(question=question)
  try:
    response = client.models.generate_content(
      model = 'gemini-2.5-flash',
      contents=prompt,
      config={'temperature': 0.1, 'response_mime_type': 'application/json'}
    )
    result = json.loads(response.text)
    return result

  except Exception as e:
    return {
      'cypher_query': None,
      'explanation': f"Error: {str(e)}",
      'expected_output': None
    }

def execute_cypher_query(graph, cypher_query: str) -> List[Dict[str, Any]]:
  try:
    result = graph.query(cypher_query)
    records = []

    for record in result:
      records.append(dict(record))

    return records

  except Exception as e:
      print(f"Error executing Cypher query: {e}")
      return []

def query_chatbot(question: str, neo4j_session) -> Dict[str, Any]:
  # Generate Cypher query
  query_info = generate_cypher_query(question)
  if not query_info['cypher_query']:
    return {
      'question': question,
      'cypher_query': None,
      'results': [],
      'answer': 'N/A'
    }

  # Execute query
  results = execute_cypher_query(neo4j_session, query_info['cypher_query'])

  return {
    'question': question,
    'cypher_query': query_info['cypher_query'],
    'results': results,
  }

def query_to_csv(query_response: Dict[str, Any], filename: str = 'query_results.csv'):
  question = query_response.get('question', 'N/A')
  cypher_query = query_response.get('cypher_query', 'N/A')
  results = query_response.get('results', [])

  # Check if file exists to determine if we need to write header
  file_exists = os.path.isfile(filename)

  with open(filename, 'a', newline='', encoding='utf-8') as f:
      writer = csv.writer(f)

      # Write header only if file doesn't exist
      if not file_exists:
          writer.writerow(['Question', 'Cypher Query', 'Result'])

      if not results:
          # No results case
          writer.writerow([question, cypher_query, 'No results found'])
      else:
          # Merge all results
          merged_results = []
          for result in results:
              # Check for standard evidence/source fields
              evidence = result.get('Evidence', result.get('source_text', None))
              source_url = result.get('SourceURL', result.get('source_url', None))

              # Check for aggregation fields (arrays)
              sample_evidence = result.get('SampleEvidence', None)
              sample_urls = result.get('SampleSourceURLs', None)

              if evidence and source_url:
                  # Standard query result
                  merged_results.append(f"{evidence} from sourceURL: {source_url}")
              elif sample_evidence and sample_urls:
                  # Aggregation query with arrays - pair them up
                  for ev, url in zip(sample_evidence, sample_urls):
                      if ev and url:
                          merged_results.append(f"{ev} from sourceURL: {url}")
              else:
                  # Fallback: create summary from all non-standard fields
                  summary_parts = []
                  for key, value in result.items():
                      if key not in ['Evidence', 'SourceURL', 'source_text', 'source_url', 'SampleEvidence', 'SampleSourceURLs']:
                          summary_parts.append(f"{key}: {value}")
                  if summary_parts:
                      merged_results.append(' | '.join(summary_parts))

          # Join all results with newline
          final_result = '\n'.join(merged_results) if merged_results else 'No evidence found'

          # Write single row with all results
          writer.writerow([question, cypher_query, final_result])

In [17]:
import csv
from model import Model
if __name__ == "__main__":
    # 1. Download dataset from huggingface
    # file_path = hf_hub_download(
    #     repo_id="Zihan1004/FNSPID",
    #     filename="Stock_news/nasdaq_exteral_data.csv",
    #     repo_type="dataset"
    # )
    # sample = filtered_text(file_path)
    #with open('/Users/anhvu/Desktop/Submission/IDM/Assignment3/sample-text', 'r') as f: # hard-coded for POC development
    #    sample = f.read()

    # 2. Load and filter CSV data for AAPL on 2023-12-01 using financial_ontology

    # Load CSV and filter for AAPL on 2023-12-01
    file_path = 'nasdaq_100.csv'
    df = pd.read_csv(
        file_path,
        dtype=str,
        low_memory=False,
        encoding='utf-8',
        on_bad_lines='skip',
        sep=',',
        quotechar='"',
        quoting=csv.QUOTE_MINIMAL
    )

    # Filter for AAPL on 2023-12-01
    df['Date'] = pd.to_datetime(df['Date'])
    target_date = pd.to_datetime('2023-12-01').date()
    df_filtered = df[
        (df['Date'].dt.date == target_date) &
        (df['Stock_symbol'] == 'AAPL')
    ].copy() # Added .copy() to prevent SettingWithCopyWarning

    if df_filtered.empty:
        print(f"No data found for AAPL on 2023-12-01")
        triplets = []
    else:
        # Create the 'information' column using join_string function
        df_filtered['Information'] = df_filtered[
            ['Date', 'Article_title', 'Stock_symbol', 'Url', 'Lexrank_summary']
        ].apply(join_string, axis=1)

        # Group all information and return as text
        df_grouped = df_filtered.groupby('Date')['Information'].apply(
            lambda x: '\n'.join(x)
        ).reset_index()

        triplets = []

        print(f"Extracted text from {len(df_filtered)} articles for AAPL on 2023-12-01")
        for idx, row in df_filtered.iterrows():
            sample_text = row['Information']

            # Use LLMs to extract kg entities and relationship
            extracted_triplets = extract_entities_and_relationship(sample_text, Model)
            if isinstance(extracted_triplets, dict):
                triplets.append(extracted_triplets) # Wrap in a list if it's a single dict
            elif isinstance(extracted_triplets, list):
                triplets.extend(extracted_triplets)

        print(f"Extracted {len(triplets)} triplets using financial_ontology structure")

    # 3. Add relationships to Neo4j Instance
    load_dotenv()
    graph = Neo4jGraph(
        url=userdata.get('NEO4J_URI'),
        username=userdata.get('NEO4J_USERNAME'),
        password=userdata.get('NEO4J_PASSWORD'),
        database=userdata.get('NEO4J_DATABASE')
    )
    clear_kg(graph)
    add_relationship_to_neo4j(graph, triplets)

    # 4. LLMs output from few-shot guiding questions (10 questions)
    questions = [
        "What products does Apple produce?",
        "How much did AAPL stock increase?",
        "Who are Apple's competitors?",
        "What events affected Apple's stock?",
        "What products does Apple produce and which events increased their sales?",
        "How many relationships does Apple have in the graph?",
        "What happened with Apple in December 2023?"
    ]
    results_filename = 'results_p2.csv'
    for question in questions:
        response = query_chatbot(
            question=question,
            neo4j_session=graph
        )
        query_to_csv(response, results_filename)
    graph.close()

    # 5. Run LLM as a judge on input and output, compare with groundtruth answer done by human

Extracted text from 20 articles for AAPL on 2023-12-01
Extracted 20 triplets using financial_ontology structure
Error adding relationship: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'designated': expected a parameter, '&', '*', ':', 'WHERE', ']', '{' or '|' (line 1, column 155 (offset: 154))
"MERGE (a:Entity {name: $source}) ON CREATE SET a.type = $source_type MERGE (b:Entity {name: $target}) ON CREATE SET b.type = $target_type MERGE (a)-[r:is designated as a gatekeeper under]->(b) SET r.date = $date, r.ticker = $ticker, r.source_url = $source_url, r.source_text = $source_text"
                                                                                                                                                           ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}
Error adding relationship: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'Advertising'

In [16]:

questions = [
    "What products does Apple produce?",
    "How much did AAPL stock increase?",
    "Who are Apple's competitors?",
    "What events affected Apple's stock?",
    "What products does Apple produce and which events increased their sales?",
    "How many relationships does Apple have in the graph?",
    "What happened with Apple in December 2023?"
]

results_filename = 'results.csv'
for question in questions:
    response = query_chatbot(
        question=question,
        neo4j_session=graph
    )
    query_to_csv(response, results_filename)
graph.close()

# Task
Export the entire Neo4j graph data, including all nodes and relationships with their properties, into a JSON file named "neo4j_graph_export.json".

## Initialize Neo4j Connection

### Subtask:
Initialize the Neo4j connection using the environment variables.


**Reasoning**:
The subtask requires initializing the Neo4j connection by uncommenting and executing the relevant lines in cell `0fa86645`. I will modify cell `0fa86645` to uncomment the Neo4j initialization code.



**Reasoning**:
The Neo4j connection has already been initialized in the previous step by uncommenting and executing the relevant lines in cell `996429cf`. I will verify the connection by printing the graph object.



In [ ]:
print(graph)


**Reasoning**:
To export the entire Neo4j graph, I need to retrieve all nodes and relationships along with their properties. I will use Cypher queries to fetch this data from the Neo4j instance.



**Reasoning**:
The previous code block produced `DeprecationWarning` messages because `id` is deprecated for nodes and relationships; the warnings suggest using `element_id` instead. I will modify the code to use `element_id` for both nodes and relationships to resolve these warnings.



In [ ]:
import json

def export_neo4j_graph_to_json(kg, filename="neo4j_graph_export.json"):
    graph_data = {
        "nodes": [],
        "relationships": []
    }

    with kg._driver.session() as session:
        # Retrieve all nodes
        nodes_result = session.run("MATCH (n) RETURN n")
        for record in nodes_result:
            node = record["n"]
            graph_data["nodes"].append({
                "id": node.element_id, # Changed from node.id
                "labels": list(node.labels),
                "properties": dict(node.items())
            })

        # Retrieve all relationships
        relationships_result = session.run("MATCH (n)-[r]->(m) RETURN n, r, m")
        for record in relationships_result:
            start_node = record["n"]
            relationship = record["r"]
            end_node = record["m"]
            graph_data["relationships"].append({
                "id": relationship.element_id, # Changed from relationship.id
                "type": relationship.type,
                "start_node_id": start_node.element_id, # Changed from start_node.id
                "end_node_id": end_node.element_id, # Changed from end_node.id
                "properties": dict(relationship.items())
            })

    with open(filename, 'w') as f:
        json.dump(graph_data, f, indent=4)
    print(f"Neo4j graph data exported to {filename}")

# Call the function with the initialized graph object
export_neo4j_graph_to_json(graph)

Neo4j graph data exported to neo4j_graph_export.json


## Final Task

### Subtask:
Confirm the successful export of the graph data to a JSON file and provide instructions on how to access or view the exported file.


## Summary:

### Q&A
The successful export of the graph data to a JSON file, named "neo4j_graph_export.json", has been confirmed. To access or view the exported file, locate the `neo4j_graph_export.json` file in the current working directory of the execution environment. This file can be opened and viewed with any standard text editor or a JSON viewer, which will display the graph's nodes and relationships along with their properties in a structured JSON format.

### Data Analysis Key Findings
*   The Neo4j database connection was successfully initialized, allowing interaction with the graph database.
*   An initial attempt to export graph data encountered deprecation warnings for using `node.id` and `relationship.id`, indicating that `element_id` should be used instead for unique identification.
*   The export script was modified to correctly use `element_id` for nodes and relationships, resolving the deprecation warnings.
*   The entire Neo4j graph data, comprising all nodes and relationships with their respective properties, was successfully exported to a JSON file named "neo4j_graph_export.json".

### Insights or Next Steps
*   Always use `element_id` instead of `id` when accessing node and relationship identifiers in Neo4j to ensure compatibility with future driver versions and avoid deprecation warnings.
*   The exported JSON file can now be used for external analysis, visualization, or integration with other systems that consume graph data in a JSON format.
